# SETU-RAG on Kaggle
Code-switch-aware multilingual **speech-to-speech** RAG for 22 Indian languages.

**Before running:** Settings → Accelerator → **GPU T4 ×2**, and Settings → **Internet → On**.
See `KAGGLE.md` for full details.

## 1 · Bring the code in
Upload `setu-rag.zip` as a Kaggle **Dataset**, then **Add Input** it (mounts at `/kaggle/input/...`).

In [ ]:
import os, zipfile, shutil
CANDIDATES = ['/kaggle/input/setu-rag/setu-rag.zip', '/kaggle/input/setu-rag/setu-rag']
dst = '/kaggle/working/setu-rag'
for src in CANDIDATES:
    if os.path.isfile(src):
        zipfile.ZipFile(src).extractall('/kaggle/working/'); break
    if os.path.isdir(src):
        shutil.copytree(src, dst, dirs_exist_ok=True); break
os.chdir(dst if os.path.isdir(dst) else '/kaggle/working')
print('cwd:', os.getcwd()); print(os.listdir('.')[:12])

## 2 · Hugging Face token (Add-ons → Secrets → `HF_TOKEN`)
Needed for gated/rate-limited models. Skip if you only run the implemented logic in step 4.

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    import os
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    os.environ['HF_HOME'] = '/kaggle/working/hf_cache'   # persist across Save Version
    from huggingface_hub import login; login(os.environ['HF_TOKEN'])
    print('HF login OK')
except Exception as e:
    print('Skipping HF login (set the HF_TOKEN secret to enable downloads):', e)

## 3 · Install dependencies  *(Internet must be On)*

In [ ]:
!pip -q install -r requirements.txt
# from-source extras:
# !pip -q install git+https://github.com/huggingface/parler-tts.git
# IndicConformer (NeMo) / IndicTrans2 toolkit / IndicLID: see their GitHub READMEs

## 4 · Smoke test — the implemented novel logic (no downloads)

In [ ]:
import sys; sys.path.insert(0, os.getcwd())
from setu_rag.front_end.language_id import LanguageIdentifier
from setu_rag.front_end.cmi import compute_cmi
from setu_rag.router.adaptive_router import decide_route
lid = LanguageIdentifier().load()
for q in ['mera refund kab tak aayega order cancel kiya tha',
          'why is my coupon not applying to the cart']:
    t = lid.tag(q); c = compute_cmi(t); fr = sum(x.script=='Latn' for x in t)/len(t)
    print(f'CMI={c.cmi:.2f} matrix={c.matrix_lang} -> {decide_route(c, fr).route.value}  | {q[:40]}')

## 5 · Speech-layer logic (LID fusion + CMI-conditioned TTS description)

In [ ]:
from setu_rag.speech.lid_fusion import fuse
from setu_rag.speech.tts import style_description
from setu_rag.eval.speech_metrics import wer, cer
toks = lid.tag('mera refund kab tak aayega order cancel kiya')
print(fuse({'hin':0.8,'eng':0.2}, toks, alpha=0.4))
print(style_description('hin_Deva', 0.37))
print('WER', wer('mera refund kab aayega','mera refund kab ayega'))

## 6 · Full pipeline (after wiring the `# TODO` model stubs)
Each stub documents its exact model call. Then:

In [ ]:
# !python scripts/build_index.py --faqs data/faqs.sample.jsonl --out /kaggle/working/index
# from setu_rag.pipeline import SetuRAG
# from setu_rag.speech_pipeline import SpeechSetuRAG
# voice = SpeechSetuRAG(SetuRAG(index=..., translator=...))
# turn = voice.answer_file('question.wav', '/kaggle/working/reply.wav')
# print(turn.transcript, '->', turn.answer_text, turn.timings_ms)

## 7 · Voice demo (public link with working mic)

In [ ]:
# in scripts/serve_voice.py change demo.launch() -> demo.launch(share=True)
# !python scripts/serve_voice.py